# PneumoniaMNIST — guided self-learning notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaiKiranKumarNalla/Medical-AI-self-learning-tasks/blob/main/pneumoniamnist/pneumoniamnist_study.ipynb)

AI-assisted educational implementation with learner-led interpretation. Work cell by cell: read, predict, run, inspect, discuss. This notebook is a new teaching run, not the source of the historical checkpoint results or an exact paper replication. Training and final test evaluation have explicit switches below.

Reference: [MedMNIST v2](https://doi.org/10.1038/s41597-022-01721-8). The dataset contains 28×28 grayscale radiographs, labels 0=normal and 1=pneumonia. Image counts are not verified patient counts. Eight exact image hashes overlap between official train and validation; this notebook removes those validation examples for new experiments. Historical test-guided exploration remains exploratory.

Colab runtime files are temporary. Copy your selected checkpoints and configuration to durable storage before disconnecting; the saved notebook alone does not preserve them. No conversation history is included.


## 1. Environment Setup & Hardware Acceleration

In [ ]:
# Install dependencies if running on Google Colab or fresh environment
try:
    import medmnist
except ImportError:
    !pip install -q medmnist scikit-learn matplotlib torchvision torch

import os
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn.utils.prune as prune
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

# Automatic Hardware Detection (CUDA -> MPS -> CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🚀 Using NVIDIA CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🚀 Using Apple Silicon GPU (MPS)")
else:
    device = torch.device('cpu')
    print("ℹ️ Using CPU")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

from pathlib import Path
import hashlib
import copy
from datetime import datetime
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.makedirs("data", exist_ok=True)


## 2. Data Loading & Class Distribution Analysis
PneumoniaMNIST is derived from pediatric chest radiographs (ages 1-5, Guangzhou Women and Children's Medical Center).
The dataset has an inherent class imbalance in the training set: **74.2% pneumonia vs 25.8% normal**.

In [ ]:
import medmnist
from medmnist import PneumoniaMNIST, INFO

data_flag = 'pneumoniamnist'
info = INFO[data_flag]
print("Dataset Info:")
print(f"  Modality: {info['label']}")
print(f"  Description: {info['description']}")
print(f"  Channels: {info['n_channels']}, Classes: {len(info['label'])}")

# Standard PyTorch normalization for MedMNIST: [0, 1] -> [-1, 1]
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True, root='data')
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True, root='data')
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True, root='data')

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# Remove exact training-image matches from validation for this teaching run.
train_hashes = {hashlib.sha256(img.tobytes()).digest() for img in train_dataset.imgs}
keep = [i for i, img in enumerate(val_dataset.imgs)
        if hashlib.sha256(img.tobytes()).digest() not in train_hashes]
val_clean = torch.utils.data.Subset(val_dataset, keep)
print(f"Validation: {len(val_dataset)} official -> {len(val_clean)} after deduplication")
val_loader = DataLoader(val_clean, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Analyze Split Distributions
def analyze_split(dataset, name):
    labels = dataset.labels.flatten()
    n_total = len(labels)
    n_norm = (labels == 0).sum()
    n_pneu = (labels == 1).sum()
    print(f"{name:5s} -> Total: {n_total:4d} | Normal: {n_norm:4d} ({n_norm/n_total*100:.1f}%) | Pneumonia: {n_pneu:4d} ({n_pneu/n_total*100:.1f}%)")
    return n_norm, n_pneu

print("\nClass Distributions:")
tr_n, tr_p = analyze_split(train_dataset, "Train")
va_n, va_p = analyze_split(val_dataset, "Val")
te_n, te_p = analyze_split(test_dataset, "Test")

# Imbalance Ratio for Class-Weighted Loss
weight_normal = tr_p / tr_n
print(f"\nInverse Class Ratio for Normal Class: {weight_normal:.3f}x")


## 3. Exploratory Data Visualization
Inspecting sample Normal vs. Pneumonia chest radiographs.

In [ ]:
# Denormalize image for display: [-1, 1] -> [0, 1]
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1).numpy()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
norm_count, pneu_count = 0, 0

for img, label in train_dataset:
    lbl = int(label[0])
    if lbl == 0 and norm_count < 6:
        ax = axes[0, norm_count]
        ax.imshow(denorm(img[0]), cmap='gray')
        ax.set_title("Normal (0)", fontsize=10, color='navy')
        ax.axis('off')
        norm_count += 1
    elif lbl == 1 and pneu_count < 6:
        ax = axes[1, pneu_count]
        ax.imshow(denorm(img[0]), cmap='gray')
        ax.set_title("Pneumonia (1)", fontsize=10, color='darkred')
        ax.axis('off')
        pneu_count += 1
    if norm_count == 6 and pneu_count == 6:
        break

plt.suptitle("PneumoniaMNIST: Sample Radiographs (28x28)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Architectures: small-image ResNet and PlainNet
The 3×3 stride-1 stem preserves 28×28 resolution. Residual blocks add a shortcut; projection shortcuts match changed channel counts. PlainNet omits those shortcuts and has fewer parameters. This is not a parameter-matched ablation. Trace input → convolutions → pooling → two logits; logits are not probabilities.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet18(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super(ResNet18, self).__init__()
        self.in_planes = 64
        # Author 3x3 stride-1 stem (no maxpool)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(BasicBlock, 64,  2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.linear(out)


class PlainBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(PlainBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # RESIDUAL ADDITION INTENTIONALLY REMOVED FOR ABLATION!
        # out += self.shortcut(x) <-- SEVERED
        return F.relu(out)

class PlainNet18(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super(PlainNet18, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(PlainBlock, 64,  2, stride=1)
        self.layer2 = self._make_layer(PlainBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(PlainBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(PlainBlock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.linear(out)


def create_resnet18(num_classes=2):
    return ResNet18(num_classes=num_classes)

for cls in (ResNet18, PlainNet18):
    example = cls().eval()
    with torch.no_grad(): shape = example(torch.zeros(1, 1, 28, 28)).shape
    print(cls.__name__, sum(p.numel() for p in example.parameters()), tuple(shape))
del example


## 5. Loss Function Formulations
To tackle the class imbalance ($74.2\%$ pneumonia), we implement:
- **Standard Cross-Entropy**
- **Class-Weighted Cross-Entropy** ($w_0 = 2.878, w_1 = 1.0$)
- **Focal Loss** (Lin et al. 2017):
$$\mathcal{L}_{\text{Focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
where $\gamma=2.0$ downweights easy confident predictions, focusing gradients on hard boundary examples.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # Tensor of shape (num_classes,) or None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: (B, C), targets: (B,)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of true class
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# Helper function to get desired loss
def get_loss_function(strategy='standard', device='cpu'):
    if strategy == 'standard':
        return nn.CrossEntropyLoss()
    elif strategy == 'weighted':
        weights = torch.tensor([2.878, 1.0], dtype=torch.float32).to(device)
        return nn.CrossEntropyLoss(weight=weights)
    elif strategy == 'focal':
        alpha = torch.tensor([0.742, 0.258], dtype=torch.float32).to(device)
        return FocalLoss(alpha=alpha, gamma=2.0)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")


## 6. Evaluation
AUC measures ranking; sensitivity/specificity use a fixed probability threshold of 0.5. These benchmark metrics do not establish clinical utility.


In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    all_targets = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.reshape(-1).long().to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)[:, 1]
            preds = (probs >= 0.5).long()

            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    auc = roc_auc_score(all_targets, all_probs) if len(np.unique(all_targets)) == 2 else float('nan')
    acc = (all_preds == all_targets).mean()

    tn, fp, fn, tp = confusion_matrix(all_targets, all_preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        'auc': float(auc),
        'acc': float(acc),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'targets': all_targets,
        'probs': all_probs,
        'preds': all_preds
    }


## 7. Modular Training Engine with Early Stopping
Monitors Validation AUC and saves the checkpoint achieving peak validation performance.

In [ ]:
def train_experiment(model, train_loader, val_loader, loss_fn, optimizer, scheduler,
                     device, epochs=25, patience=7, save_path='best_model.pth'):
    model = model.to(device)
    if epochs < 1: raise ValueError('epochs must be positive')
    if Path(save_path).exists(): raise FileExistsError(save_path)
    best_val_auc = -float('inf')
    best_epoch = 0
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_auc': [], 'val_acc': []}

    print(f"Starting training on {device} (Max Epochs: {epochs}, Early Stopping Patience: {patience})...")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_weight = 0.0
        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.reshape(-1).long().to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()

            # Match the reduction used by weighted cross-entropy.
            weight = (loss_fn.weight[targets].sum().item()
                      if isinstance(loss_fn, nn.CrossEntropyLoss) and loss_fn.weight is not None
                      else targets.numel())
            running_loss += loss.item() * weight
            total_weight += weight

        if scheduler is not None:
            scheduler.step()

        epoch_loss = running_loss / total_weight
        val_metrics = evaluate_model(model, val_loader, device)

        history['train_loss'].append(epoch_loss)
        history['val_auc'].append(val_metrics['auc'])
        history['val_acc'].append(val_metrics['acc'])

        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {epoch_loss:.4f} | Val AUC: {val_metrics['auc']:.4f} | Val Acc: {val_metrics['acc']*100:.2f}% | Val Spec: {val_metrics['specificity']*100:.2f}%")

        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹️ Early stopping triggered at epoch {epoch}. Best Val AUC: {best_val_auc:.4f} at epoch {best_epoch}.")
                break

    # Load best checkpoint
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    return model, history


## 8. Optional training: choose an experiment
Start with a short smoke run, then agree on a budget. `RUN_TRAINING=False` prevents accidental training when inspecting the notebook. Choose baseline, weighted, focal or plain; compare equal budgets and validation selection rules. A short run does not reproduce historical results. Repeated execution creates a new checkpoint directory.


In [ ]:
RUN_TRAINING = False
EXPERIMENT = "focal"  # baseline, weighted, focal, plain
EPOCHS = 5  # Teaching budget, not the historical 100-epoch baseline.
model = None
if RUN_TRAINING:
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    run_dir = Path("runs") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    run_dir.mkdir(parents=True, exist_ok=False)
    model = (PlainNet18() if EXPERIMENT == "plain" else ResNet18()).to(device)
    loss_fn = (nn.CrossEntropyLoss() if EXPERIMENT in ("baseline", "plain")
               else get_loss_function(EXPERIMENT, device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[50, 75], gamma=0.1)
    model, history = train_experiment(model, train_loader, val_loader, loss_fn,
                                      optimizer, scheduler, device, epochs=EPOCHS,
                                      patience=10, save_path=str(run_dir / "best.pth"))
    import json
    (run_dir / "config.json").write_text(json.dumps({"experiment":EXPERIMENT,"epochs":EPOCHS,
        "seed":42,"validation":"exact train overlaps removed","device":str(device)},indent=2))
    (run_dir / "history.json").write_text(json.dumps(history, indent=2))
    validation_results = evaluate_model(model, val_loader, device)
    print({k:v for k,v in validation_results.items() if k not in ("targets","probs","preds")})
else:
    print("Training not started. Discuss the experiment, then set RUN_TRAINING=True.")


## 9. Optional pruning — validation only
Explore sparsity on independent copies of the selected model. This changes weights, not dense tensor dimensions; speed and storage improvements require measurement. Select any pruning setting using validation only. No historical pruning score is claimed here.


In [ ]:
RUN_PRUNING = False
if RUN_PRUNING:
    if model is None: raise RuntimeError("Train/load a model first.")
    for amount in [0.0, 0.2, 0.5, 0.8]:
        candidate = copy.deepcopy(model)
        params = [(module, "weight") for module in candidate.modules()
                  if isinstance(module, (nn.Conv2d, nn.Linear))]
        if amount:
            prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
        metrics = evaluate_model(candidate, val_loader, device)
        print(amount, {k:metrics[k] for k in ["auc","acc","sensitivity","specificity"]})
        del candidate
else:
    print("Pruning not started.")


## 10. Grad-CAM on validation examples
Explain the pneumonia logit consistently. Select TP/TN/FP/FN from actual labels/predictions; some categories may be absent. Heatmaps are coarse feature attributions, not lesion masks or proof of a clinical reason for an error. A zero map can occur and must not be turned into a fabricated explanation.


In [ ]:
def grad_cam(model, image):
    model.eval()
    features = []
    handle = model.layer4.register_forward_hook(lambda module, inputs, output: features.append(output))
    try:
        logits = model(image)
        activation = features[0]
        gradients = torch.autograd.grad(logits[0, 1], activation)[0]
        cam = F.relu((gradients.mean((2, 3), keepdim=True) * activation).sum(1, keepdim=True))
        cam = F.interpolate(cam, image.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam.detach().cpu().numpy()
        if cam.max() > 0: cam = cam / cam.max()
        return cam
    finally:
        handle.remove()

if model is not None:
    metrics = evaluate_model(model, val_loader, device)
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, (name, truth, prediction) in zip(axes, [("TP",1,1),("TN",0,0),("FP",0,1),("FN",1,0)]):
        matches = np.flatnonzero((metrics["targets"] == truth) & (metrics["preds"] == prediction))
        ax.axis("off")
        if not len(matches):
            ax.set_title(name + ": none in validation")
            continue
        i = int(matches[0]); image, label = val_clean[i]
        cam = grad_cam(model, image.unsqueeze(0).to(device))
        ax.imshow(denorm(image[0]), cmap="gray")
        ax.imshow(cam, cmap="jet", vmin=0, vmax=1, alpha=.35)
        ax.set_title(f"{name}; y={truth}, pred={prediction}\nValidation index {i}")
    plt.tight_layout(); plt.show()
else:
    print("Train/load a model before attribution analysis.")


## 11. Historical results and final evaluation
The verified historical checkpoint results are in [the repository README](https://github.com/SaiKiranKumarNalla/Medical-AI-self-learning-tasks/tree/main/pneumoniamnist). They were not produced by this notebook. Historical AUCs: baseline 0.9480, weighted 0.9551, focal 0.9511, plain 0.9675. Historical test-guided exploration, duplicate validation images and unequal budgets limit conclusions.

Freeze decisions before enabling the final test cell below. This familiar benchmark is not fresh clinical confirmation. Do not tune pruning, thresholds or architecture against these test results. Save outputs and your interpretation; avoid claiming clinical superiority or exact replication.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if model is None: raise RuntimeError("Train/load the frozen selected model first.")
    final_results = evaluate_model(model, test_loader, device)
    print({k:v for k,v in final_results.items() if k not in ("targets","probs","preds")})
else:
    print("Test evaluation withheld.")
